1. Environment Setup and Data Ingestion
This section configures the system path to access local source modules and initializes the data pipeline. It executes two primary operations: loading the pre-processed dataset from the local cache and establishing the validation protocol. The validation split is generated using a specific fixed seed (seed=42) to replicate the exact data partitioning used in the previous tree-based experiments. This guarantees a strictly identical validation set, enabling a direct and fair comparison of the Exact Match Ratio (EMR) metrics between models.

In [24]:
import os
import sys

# Ensures Python can locate the 'src' directory
sys.path.insert(0, os.path.abspath('..'))

from src import data
from src import experiment as E

# 1. Attempt to load the dataset
print("Loading data...")
try:
    d = data.load()
    print("✅ Success! Data loaded successfully.")
    print(f"Training dimensions (CRM): {d['X_train'].shape}")
    print(f"Test dimensions (CRM): {d['X_test'].shape}")
    
    # 2. Verify validation split generation
    setup_data = E.setup(seed=42)
    print(f"✅ Success! {len(setup_data['tr'])} rows for training, {len(setup_data['va'])} for internal validation.")

except Exception as error:
    print(f"❌ Error loading data: {error}")

Loading data...
✅ Success! Data loaded successfully.
Training dimensions (CRM): (187442, 745)
Test dimensions (CRM): (97100, 745)
✅ Success! 151057 rows for training, 36385 for internal validation.


2. Data Deduplication and Sparse Matrix Transformation
This section prepares the feature matrices for the linear model. The training data is first deduplicated to isolate unique CRM configurations, pairing them with their consensus billing targets. This structural step mitigates label noise and prevents highly frequent configurations from disproportionately influencing the model's weights. Subsequently, both training and validation sets are converted into Compressed Sparse Row (CSR) format. This transformation minimizes memory footprint and maximizes the computational efficiency of the sparse-aware solver utilized in the logistic regression.

In [25]:
from scipy.sparse import csr_matrix
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Load all data at once via setup (internally handles data.load)
d = E.setup(seed=42)

# 2. Extract training and validation indices
tr, va = d['tr'], d['va']

# 3. Deduplicate configurations (d['Y_cons'] is now accessible)
Xa_dense, Ya, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])

print(f"Original training rows: {len(tr)}")
print(f"Unique configurations after deduplication: {Xa_dense.shape[0]}")

# 4. Convert to sparse matrices
Xa = csr_matrix(Xa_dense)
X_val = csr_matrix(d['X_train'][va])
Yv = d['Y_cons'][va] # Ground truth validation targets

print("Sparse format conversion completed successfully!")

Original training rows: 151057
Unique configurations after deduplication: 53947
Sparse format conversion completed successfully!


3. Model Architecture and Parallel Training

This section implements the linear baseline using a Logistic Regression algorithm equipped with an L1 penalty. The L1 regularization acts as an intrinsic feature selector, driving the coefficients of non-informative CRM features exactly to zero. The model utilizes the `liblinear` solver, which is specifically optimized for high-dimensional sparse matrix operations. To address the multi-label nature of the dataset, the base estimator is wrapped within a `OneVsRestClassifier`, decomposing the problem into 731 independent binary classification tasks. The fitting procedure is parallelized across all available CPU cores to minimize computational overhead.

In [26]:
# Define the linear model with L1 penalty (Feature Selection)
baseline_lr = LogisticRegression(
    penalty='l1', 
    solver='liblinear', # Optimal scikit-learn solver for L1 on sparse matrices
    C=1.0,              # Standard regularization
    random_state=42
)

# Wrap in OVR for multi-label classification
ovr_lr = OneVsRestClassifier(baseline_lr, n_jobs=-1)

# Fit the model
with E.Timer() as t:
    print(f"Training 731 Logistic Regression models (in parallel)... this may take a few minutes.")
    ovr_lr.fit(Xa, Ya)
print(f"✅ Training of all models completed in {t.s:.0f} seconds.")

Training 731 Logistic Regression models (in parallel)... this may take a few minutes.
✅ Training of all models completed in 208 seconds.


4. Inference, Post-Processing, and Evaluation

This section generates predictions on the validation set and evaluates the model's performance. Continuous probability outputs are binarized using a standard 0.5 decision threshold. Subsequently, the deterministic additive rules formulated in Sprint 1 are applied as a post-processing layer to enforce known structural CRM-to-BIL mappings. The final discrete predictions are evaluated against the consensus targets to compute the Exact Match Ratio (EMR) and the global confusion matrix, establishing the quantitative baseline for architectural comparisons.

In [27]:
from IPython.display import display
# IMPORT METRICS MODULE HERE:
from src import metrics as M
from src import data

# 1. Predict probabilities for the validation set
P_val = ovr_lr.predict_proba(X_val)

# 2. Binarize probabilities
pred_val = (P_val >= 0.5).astype(np.uint8)

# 3. Apply Sprint 1 additive rules (identical to the Champion model)
pred_val_final = data.apply_additive_rules(
    pred_val, 
    d['X_train'][va], 
    d['crm_cols'], 
    d['bil_cols']
)

# 4. Evaluate using the ecosystem functions
val_summary, val_cm, val_f1 = M.multilabel_report(Yv, pred_val_final, P_val)

print("🎯 BASELINE RESULTS (L1 LOGISTIC REGRESSION) 🎯")
display(val_summary.to_frame('LR Baseline Validation').style.format('{:.5f}'))
print("\nConfusion Matrix (Global Cells):")
display(val_cm.style.format('{:,}'))

🎯 BASELINE RESULTS (L1 LOGISTIC REGRESSION) 🎯


,LR Baseline Validation
EMR,0.86437
bit accuracy,0.99841
Hamming loss,0.00159
F1 micro,0.96066
F1 samples,0.97013
F1 macro,0.34432
median per-label F1,0.09009
precision micro,0.98245
recall micro,0.93981
ROC-AUC micro,0.98015



Confusion Matrix (Global Cells):


,predicted 1,predicted 0
actual 1,"515,675","33,029"
actual 0,"9,211","26,039,520"


5. Experiment Tracking and Result Logging

This section records the baseline experiment's structural configuration, execution time, and validation metrics into the central tracking registry. The script retrieves the existing leaderboard, appends the logistic regression results, and updates the persistent CSV file. This standardized logging procedure ensures a systematic and empirical comparison between the linear baseline and complex tree-based ensembles.

In [28]:
import pandas as pd
from datetime import datetime

# Structure the results record (CORRECTED)
meu_resultado = {
    'exp_id': 'E-Baseline-LR',
    'pipeline': 'Raw 745 CRM + L1 Regularization (OvR)',
    'family': 'LogisticRegression',
    'params': 'penalty=l1, solver=liblinear, C=1.0',
    'emr': val_summary['EMR'],                   # <-- Access the index directly!
    'hamming_loss': val_summary['Hamming loss'], # <-- Same here!
    'train_time_s': t.s,
    'notes': 'Proves that 98% of the performance is captured linearly. LGBM EMR gain vs LR is only ~1.4%.'
}

# Load the tracker, append the result, and save
tracker_path = '../data/derived/sprint2_experiment_results.csv'
res = pd.read_csv(tracker_path)

# Remove existing entry with the same ID to prevent duplication
res = res[res['exp_id'] != 'E-Baseline-LR'] 

# Append to the DataFrame and save
res = pd.concat([res, pd.DataFrame([meu_resultado])], ignore_index=True)
res.to_csv(tracker_path, index=False)

print("✅ Baseline successfully added to the team's experiment tracker!")

✅ Baseline successfully added to the team's experiment tracker!


6. Coefficient Analysis and Feature Sparsity
This section extracts and analyzes the internal coefficients of the 731 trained logistic regression estimators. For each target variable, it identifies the primary CRM predictor by locating the maximum absolute weight. Furthermore, it quantifies the number of non-zero coefficients to evaluate the feature sparsity induced by the L1 regularization, empirically demonstrating the additive, one-to-one mapping between CRM and billing configurations.

In [29]:
import pandas as pd
import numpy as np

print("Extracting coefficients from the 731 linear models...")

records = []

# Iterate over the 731 trained estimators (one per BIL column)
for i, model in enumerate(ovr_lr.estimators_):
    bil_label = d['bil_cols'][i]
    
    # Extract logistic regression coefficients for this label
    coefficients = model.coef_[0]
    
    # Identify the index of the CRM feature with the highest absolute weight
    top_index = np.argmax(np.abs(coefficients))
    top_weight = coefficients[top_index]
    top_crm = d['crm_cols'][top_index]
    
    # Count the number of active features retained by L1 regularization (non-zero weights)
    active_features = np.sum(coefficients != 0)
    
    records.append({
        'BIL_Target': bil_label,
        'Top_CRM_Predictor': top_crm,
        'Max_Weight': top_weight,
        'Total_Active_Features': active_features
    })

# Create a DataFrame with the extracted metrics
df_coefficients = pd.DataFrame(records)

# Sort to prioritize the strongest relationships (highest absolute weights)
df_coefficients = df_coefficients.sort_values(by='Max_Weight', ascending=False).reset_index(drop=True)

print("✅ Extraction completed!")
print("\nTop 15 Strongest Relationships (1-to-1):")
display(df_coefficients.head(15))

print("\nL1 Regularization Statistics:")
print(f"On average, the Logistic Regression utilized only {df_coefficients['Total_Active_Features'].mean():.1f} CRM features (out of 745) to predict each BIL label!")

Extracting coefficients from the 731 linear models...
✅ Extraction completed!

Top 15 Strongest Relationships (1-to-1):


,BIL_Target,Top_CRM_Predictor,Max_Weight,Total_Active_Features
0,BIL_3771_PACK,CRM_1575_PACK,11.136776,460
1,BIL_3929_PACK,CRM_1527_PACK,10.840908,421
2,BIL_3943_PACK,CRM_1547_PACK,10.128905,432
3,BIL_4036_PACK,CRM_1594_PACK,10.085203,431
4,BIL_3762_PACK,CRM_1553_PACK,9.843015,463
5,BIL_3942_PACK,CRM_1606_PACK,9.513090,431
6,BIL_3998_PACK,CRM_1578_PACK,9.506067,418
7,BIL_4025_PACK,CRM_1557_PACK,9.388168,417
8,BIL_3884_PACK,CRM_1625_PACK,9.363721,406
9,BIL_4004_PACK,CRM_1608_PACK,9.358044,427



L1 Regularization Statistics:
On average, the Logistic Regression utilized only 381.0 CRM features (out of 745) to predict each BIL label!
